---

# Dudley Disability Data Cleaning

---

## Purpose

Clean and validate Census 2021 disability data for Dudley residents aged
16 years and over, ready for later analysis of local employment barriers
and possible support needs.

The source is RM019 — Economic activity status by hours worked by
disability, accessed through ONS/Nomis. The economic activity selection
is Total, covering all economic activity statuses.

## Objectives

- Inspect the raw file structure and load the dataset.
- Identify and handle blank rows and other non-data rows.
- Check missing values, duplicates and LSOA identifiers.
- Standardise column names and validate numeric counts.
- Save a cleaned CSV and document the cleaning decisions.

## Inputs

- `raw_datasets/dudley_disability_16_plus.csv`
- Census 2021 data downloaded from Nomis on 16 October 2026.

## Outputs

- `cleaned_datasets/dudley_disability_16_cleaned.csv`
- Documented checks, findings and a cleaning conclusion.

## Ethical Considerations

Disability status does not establish a person's ability or willingness
to work, or their individual support needs. These neighbourhood statistics
must not be used to make assumptions about individual residents.

The population includes everyone aged 16 and over, including older
residents. It must not be described as exclusively working age.

## Additional Comments

The raw source file will remain unchanged during cleaning.
Percentage calculations, analysis, visualisations and modelling will
be completed in separate notebooks.

Code developed with assistance from ChatGPT will be acknowledged,
with relevant documentation linked in comments.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

### Import libraries

Import pandas to load, inspect and clean the disability dataset.

In [4]:
# Import pandas for loading and cleaning the dataset.
import pandas as pd

## 1. Inspect the raw file structure

Read the first ten lines to check the dataset title, population and
column-heading row before loading the CSV with pandas.

## Ethical Considerations When Inspecting the Raw Disability Dataset

Inspecting the first ten lines of the disability dataset is an essential ethical step before loading the CSV into pandas.  
Because this dataset describes real residents aged 16 years and over in Dudley, including sensitive information about disability status, it must be handled with care, transparency, and respect.

### Why Ethical Inspection Matters

The introductory lines of the raw file often contain:
- Dataset title  
- Population coverage  
- Units  
- Metadata  
- Copyright notes  
- Column‑heading information  

Understanding these elements is crucial for:
- **Data provenance** — confirming the dataset comes from a trusted source (ONS/Nomis).  
- **Context** — understanding what the dataset measures and who it represents.  
- **Integrity** — identifying whether non‑data rows need to be skipped to avoid corrupting the dataset.

Disability data is particularly sensitive because it relates to:
- Health limitations  
- Barriers to employment  
- Social inclusion  
- Support needs  

Misinterpreting this data can lead to unfair or harmful conclusions.

### Ethical Risks if Raw Inspection Is Ignored

If the raw file is loaded without inspection, several ethical risks arise:

- **Misinterpreting population coverage**  
  Analysts may assume the dataset covers all adults when it may only cover residents aged 16+.

- **Incorrect handling of metadata**  
  Non‑data rows could be mistaken for real records, distorting disability counts.

- **Loss of transparency**  
  Without documenting the raw structure, users cannot verify that the dataset was loaded responsibly.

- **Misleading downstream analysis**  
  Incorrect column headings or skipped lines may lead to inaccurate dashboards or unfair comparisons between neighbourhoods.

###  Ethcal Safeguards Used in This Step

This inspection process supports ethical data handling by:

- **Displaying raw lines exactly as they appear**  
  Ensures transparency and allows manual verification.

- **Avoiding any modification to the raw file**  
  Preserves the original dataset for auditing and accountability.

- **Checking for blank lines and metadata**  
  Helps determine whether introductory rows must be skipped when loading the full dataset.

- **Documenting population coverage and units**  
  Prevents misinterpretation of disability categories in later analysis.

### Why This Matters for Responsible Interpretation

Disability data is sensitive and can influence how communities are perceived.  
Incorrect handling of the raw file can lead to:

- Misleading conclusions about health limitations  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By inspecting the raw dataset carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [5]:
# Developed with assistance from ChatGPT.
raw_path = "raw_datasets/dudley_disability_16_plus.csv"

# Read the source headings without changing the raw file.
with open(raw_path, "r", encoding="utf-8-sig") as file:
    for line_number in range(1, 11):
        line = file.readline()
        if not line:
            break
        print(f"{line_number}: {line.rstrip()}")

1: RM019 - Economic activity status by hours worked by disability,,,,,,
2: ONS Crown Copyright Reserved [from Nomis on 16 October 2026],,,,,,
3: Population :,All usual residents aged 16 years and over,,,,,
4: Units      :,Persons,,,,,
5: Date       :,2021,,,,,
6: Economic activity status and hours worked:,Total,,,,,
7: ,,,,,,
8: 2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
9: ,,,,,,
10: Dudley 001A,E01009763,1640,350,154,196,1290


### Findings

- The file contains Census 2021 RM019 disability data, downloaded
  from Nomis on 16 October 2026.
- The population is all usual residents aged 16 years and over.
  Counts are measured in persons.
- The economic activity selection is Total, so the data includes
  all economic activity statuses.
- Column headings appear on line 8. Skip the first seven lines
  when loading the CSV.
- Line 9 is an empty record containing commas. This will be
  checked during cleaning.
- The table contains LSOA names, codes and five count columns.
- The two categories “limited a lot” and “limited a little” are
  subcategories of the disabled total. They must not be added
  to that total, as this would double-count residents.

---

## 2. Load and preview the dataset

Load the CSV, skipping the seven introductory lines so that line 8
provides the column headings. Preview five rows and check the initial
number of rows and columns before cleaning.

In [6]:
# Developed with assistance from ChatGPT.
# Load the data using the column headings identified during inspection.
disability_df = pd.read_csv(
    raw_path,
    skiprows=7,
    encoding="utf-8-sig"
)

# Check the initial dimensions and preview five rows.
print(f"Rows: {disability_df.shape[0]}")
print(f"Columns: {disability_df.shape[1]}")
disability_df.head()

Rows: 211
Columns: 7


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
2,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
3,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
4,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0


### Findings

- The initial dataset contains 211 rows and 7 columns.
- The column headings have loaded correctly.
- The first row contains missing values in every column, matching
  the empty record identified in the raw file.
- The following rows contain Dudley LSOA names, codes and counts.
- Counts currently display with decimal places, such as 1640.0.
  Numeric types will be checked during cleaning.
- The 211 rows are not yet a verified count of neighbourhoods.
  Blank rows and any source notes must be inspected before removal.

---

## 3. Inspect blank rows and rows without area codes

Count completely empty rows and inspect any rows without a mnemonic
(LSOA code). This helps identify blank records and source notes before
deciding which rows to remove.

## Ethical Considerations When Inspecting Blank Rows and Missing Area Codes

Inspecting blank rows and rows without an LSOA code is a crucial ethical step when preparing the disability dataset.  
Because this dataset contains sensitive information about residents aged 16 years and over — including disability status — it must be handled with accuracy, transparency, and respect for the communities represented.

### Why Ethical Inspection Matters

Blank rows and missing area codes often indicate:
- Empty records  
- Formatting artefacts  
- Source notes  
- Disclosure‑control messages  
- Corrupted or partial entries  

These rows **do not represent real neighbourhoods**, and including them in analysis could distort disability counts or misrepresent Dudley’s communities.

Disability data is particularly sensitive because it relates to:
- Health limitations  
- Barriers to employment  
- Social inclusion  
- Support needs  

Any misinterpretation can lead to unfair or harmful conclusions.

### Ethical Risks if Blank or Code‑less Rows Are Ignored

If these rows are not identified and handled correctly, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  Blank rows may be counted as real areas, inflating totals or creating false entries.

- **Incorrect disability counts**  
  Rows without LSOA codes cannot be linked to real neighbourhoods, leading to inaccurate comparisons.

- **Distorted analysis**  
  Dashboards or reports may show misleading values if non‑data rows are included.

- **Loss of transparency**  
  Users may not understand why certain rows appear empty or inconsistent.

- **Broken dataset joins**  
  Missing codes prevent accurate merging with other datasets, causing misaligned or corrupted results.

### Ethical Safeguards Used in This Step

This inspection process includes several responsible practices:

- **Detecting completely empty rows**  
  Ensures that blank records are identified before cleaning.

- **Checking for missing or whitespace‑only LSOA codes**  
  Prevents non‑data rows from being mistaken for real neighbourhoods.

- **Displaying affected rows for manual review**  
  Supports transparency and allows analysts to verify which rows should be removed.

- **Avoiding premature deletion**  
  Rows are inspected first, ensuring no valid neighbourhood is removed accidentally.

- **Preserving the raw dataset unchanged**  
  Maintains auditability and supports ethical data governance.

### Why This Matters for Responsible Interpretation

Disability data is used to:
- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Incorrect handling of blank or code‑less rows can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By inspecting blank rows and missing area codes carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [7]:
# Developed with assistance from ChatGPT.

# Identify rows where every value is missing.
blank_rows = disability_df.isna().all(axis=1)

# Identify missing or whitespace-only area codes.
missing_codes = (
    disability_df["mnemonic"].isna()
    | disability_df["mnemonic"].astype(str).str.strip().eq("")
)

print(f"Completely empty rows: {blank_rows.sum()}")
print(f"Rows without an area code: {missing_codes.sum()}")

# Show these rows for inspection before removing anything.
disability_df.loc[missing_codes]

Completely empty rows: 4
Rows without an area code: 4


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

- Four rows are completely empty: indices 0, 204, 205 and 206.
- These are also the only four rows without an area code.
- These blank records can be removed without losing population data.
- Removing them would leave 207 rows. These must still be checked
  to distinguish valid Dudley LSOAs from any other records.
- Source notes may contain text in the mnemonic column, so checking
  missing codes alone does not identify every possible non-data row.

---

## 4. Inspect other non-data records

Check for non-empty rows whose mnemonic does not match the format of
an English LSOA code: E followed by eight digits.

Inspect these records before deciding whether to remove them.
Matching the code format alone does not confirm Dudley coverage.

In [8]:
# Developed with assistance from ChatGPT.

# Check the format of each area code.
valid_code_format = (
    disability_df["mnemonic"]
    .astype("string")
    .str.fullmatch(r"E\d{8}", na=False)
)

# Identify non-empty rows without a correctly formatted code.
other_records = disability_df.loc[
    ~blank_rows & ~valid_code_format
]

print(f"Rows matching LSOA code format: {valid_code_format.sum()}")
print(f"Other non-empty records: {len(other_records)}")

# Display the records before making a cleaning decision.
other_records

Rows matching LSOA code format: 203
Other non-empty records: 4


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
207,NaN,In order to protect against disclosure of pers...,NaN,NaN,NaN,NaN,NaN
208,NaN,have been swapped between different geographic...,NaN,NaN,NaN,NaN,NaN
209,NaN,by small amounts. Small counts at the lowest g...,NaN,NaN,NaN,NaN,NaN
210,NaN,affected.,NaN,NaN,NaN,NaN,NaN


### Findings

- 203 rows have a mnemonic matching the expected LSOA code format.
- Four additional non-empty rows, at indices 207–210, contain
  disclosure-control notes rather than neighbourhood records.
- These notes appear in the mnemonic column and have no population
  counts. They can be excluded from the cleaned table while remaining
  preserved in the raw CSV.
- Together with the four blank rows, this accounts for all eight
  non-data records in the initial 211 rows.
- The remaining 203 records still require checks for duplicates,
  missing values and Dudley coverage.

---

## 5. Remove the identified non-data rows

Keep the records with correctly formatted LSOA codes, excluding the
four blank rows and four source-note rows identified above.

Create a separate cleaning dataframe and reset its row index.
The original dataframe and raw CSV remain unchanged.

## Ethical Considerations When Removing Non‑Data Rows

Removing non‑data rows is a critical ethical step in preparing the disability dataset.  
Because this dataset contains sensitive information about residents aged 16 years and over — including disability status — it is essential to ensure that only genuine neighbourhood records are retained. This protects against misinterpretation, maintains fairness, and preserves the integrity of the analysis.

### Why Ethical Care Is Needed Here

The initial dataset includes:
- 203 valid neighbourhood records with correctly formatted LSOA codes  
- 4 blank rows  
- 4 disclosure‑control note rows  

The non‑data rows do **not** represent real neighbourhoods.  
Disclosure‑control notes are included by the data provider to protect individuals’ privacy, and blank rows are structural artefacts. Including these rows in analysis would distort disability counts and misrepresent Dudley’s communities.

### Ethical Risks if Non‑Data Rows Are Not Removed

If blank rows or disclosure‑control notes are left in the dataset, several ethical risks arise:

- **Inflated or distorted disability counts**  
  Non‑data rows may be interpreted as real neighbourhoods, leading to inaccurate totals.

- **Misleading comparisons**  
  Dashboards or reports may show false neighbourhoods or incorrect values.

- **Broken dataset joins**  
  Rows without valid LSOA codes cannot be matched to other datasets, causing misalignment or corrupted results.

- **Loss of transparency**  
  Users may not understand why certain rows contain missing values or unusual text.

- **Privacy concerns**  
  Disclosure‑control notes must be preserved in the raw file but excluded from analysis to avoid misinterpretation.

### Ethical Safeguards Used in This Step

This cleaning step includes several responsible practices:

- **Filtering only rows with valid LSOA codes**  
  Ensures that every retained record corresponds to a real neighbourhood.

- **Preserving the raw dataset unchanged**  
  Maintains an audit trail and supports ethical data governance.

- **Resetting the index in the cleaned dataset**  
  Provides clarity and prevents confusion in later analysis.

- **Reporting how many rows were removed and retained**  
  Supports transparency and allows verification that only non‑data rows were excluded.

- **Previewing the first five cleaned records**  
  Allows manual confirmation that the dataset now contains only valid neighbourhoods.

### Why This Matters for Responsible Interpretation

Disability data is used to:
- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Incorrect handling of non‑data rows can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By removing non‑data rows carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [9]:
# Developed with assistance from ChatGPT.

# Keep the area records identified during inspection.
disability_clean = (
    disability_df.loc[valid_code_format]
    .copy()
    .reset_index(drop=True)
)

# Confirm how many records were removed and retained.
print(f"Rows removed: {len(disability_df) - len(disability_clean)}")
print(f"Rows remaining: {len(disability_clean)}")
print(f"Columns: {disability_clean.shape[1]}")

# Preview the first five retained records.
disability_clean.head()

Rows removed: 8
Rows remaining: 203
Columns: 7


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
1,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
2,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
3,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0
4,Dudley 001E,E01009770,1213.0,278.0,136.0,142.0,935.0


### Findings

- Eight non-data rows were removed: four blank records and four
  disclosure-control note rows.
- The cleaning dataframe now contains 203 rows and 7 columns.
- The index has been reset, and the preview begins with Dudley 001A.
- All original columns and retained population counts are preserved.
- The raw CSV and original dataframe remain unchanged.
- Further validation is needed before the dataset is saved.

---

## 6. Standardise column names

Rename the columns to make them easier to use and interpret.
Use LSOA_Name and LSOA_Code consistently with the other cleaned datasets.

Include the 16-plus population in count column names. Keep the disabled
total separate from its two activity-limitation subcategories.

In [10]:
# Developed with assistance from ChatGPT.

# Map the original headings to clear, consistent names.
column_names = {
    "2021 super output area - lower layer": "LSOA_Name",
    "mnemonic": "LSOA_Code",
    "Total": "Total_16_Plus",
    "Disabled under the Equality Act": "Disabled_16_Plus",
    (
        "Disabled under the Equality Act: "
        "Day-to-day activities limited a lot"
    ): "Limited_A_Lot_16_Plus",
    (
        "Disabled under the Equality Act: "
        "Day-to-day activities limited a little"
    ): "Limited_A_Little_16_Plus",
    "Not disabled under the Equality Act": "Not_Disabled_16_Plus"
}

disability_clean = disability_clean.rename(columns=column_names)

# Check the updated headings and preview five rows.
print(disability_clean.columns.tolist())
disability_clean.head()

['LSOA_Name', 'LSOA_Code', 'Total_16_Plus', 'Disabled_16_Plus', 'Limited_A_Lot_16_Plus', 'Limited_A_Little_16_Plus', 'Not_Disabled_16_Plus']


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus
0,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
1,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
2,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
3,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0
4,Dudley 001E,E01009770,1213.0,278.0,136.0,142.0,935.0


### Findings

- All seven columns have been renamed successfully.
- LSOA_Name and LSOA_Code provide consistent identifiers for later
  matching with the other cleaned datasets.
- The five count columns explicitly identify the population as 16-plus.
- The disabled total and its two activity-limitation subcategories
  remain separate.
- Renaming changed only the headings; the data values are unchanged.

---

## 7. Check missing values and duplicates

Check each column for missing values, then check for duplicate rows
and repeated LSOA codes. Each neighbourhood should have one record.

## Ethical Considerations When Checking Missing Values and Duplicates

Checking for missing values and duplicate records is a vital ethical step when preparing the disability dataset.  
Because this dataset contains sensitive information about residents aged 16 years and over — including disability status and activity‑limitation categories — ensuring completeness and uniqueness is essential for fair, accurate, and responsible analysis.

### Why Ethical Inspection Matters

Each neighbourhood (LSOA) should appear **exactly once** in the dataset.  
Missing values or duplicated records can distort disability counts, misrepresent communities, and lead to incorrect conclusions about health limitations or support needs.

Disability data is particularly sensitive because it relates to:
- Daily‑life limitations  
- Barriers to employment  
- Social inclusion  
- Local support requirements  

Any errors in this dataset can have real consequences for how neighbourhoods are understood and how resources are allocated.

### Ethical Risks if Missing Values or Duplicates Are Ignored

If missing values or duplicates are not identified and addressed, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  Missing disability counts may make an area appear more or less affected than it truly is.

- **Inflated or deflated totals**  
  Duplicate rows can double‑count a neighbourhood, distorting comparisons and misleading dashboards.

- **Incorrect dataset joins**  
  Repeated LSOA codes can break merges with other datasets, causing misaligned or corrupted results.

- **Loss of transparency**  
  Users may not understand why certain values appear inconsistent or contradictory.

- **Unfair policy insights**  
  Decisions based on incomplete or duplicated data may misallocate support or resources.

### Ethical Safeguards Used in This Step

This validation step includes several responsible practices:

- **Column‑level missing value checks**  
  Ensures that disability counts and identifiers are complete before analysis.

- **Checking for fully duplicated rows**  
  Confirms that no neighbourhood is represented more than once.

- **Checking for repeated LSOA codes**  
  Ensures that each neighbourhood has a unique identifier, supporting accurate dataset joins.

- **Reporting counts of missing values and duplicates**  
  Supports transparency and allows analysts to verify data integrity.

- **Maintaining the cleaned dataset separately**  
  Preserves the raw data for auditing and ethical data governance.

### Why This Matters for Responsible Interpretation

Disability data is used to:
- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Incorrect handling of missing values or duplicates can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By checking for missing values and duplicates carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [11]:
# Developed with assistance from ChatGPT.

# Count missing values in each column.
print("Missing values by column:")
print(disability_clean.isna().sum())

# Check for repeated records and area codes.
print(
    "\nDuplicate rows:",
    disability_clean.duplicated().sum()
)
print(
    "Duplicate LSOA codes:",
    disability_clean["LSOA_Code"].duplicated().sum()
)
print(
    "Unique LSOA codes:",
    disability_clean["LSOA_Code"].nunique()
)

Missing values by column:
LSOA_Name                   0
LSOA_Code                   0
Total_16_Plus               0
Disabled_16_Plus            0
Limited_A_Lot_16_Plus       0
Limited_A_Little_16_Plus    0
Not_Disabled_16_Plus        0
dtype: int64

Duplicate rows: 0
Duplicate LSOA codes: 0
Unique LSOA codes: 203


### Findings

- No missing values were found in any of the seven columns.
- There are no duplicate rows or repeated LSOA codes.
- The dataset contains 203 unique LSOA codes, with one record per code.
- No missing-value treatment or duplicate removal is needed.
- Dudley coverage will be checked separately.

---

## 8. Check Dudley neighbourhood coverage

Check that all area names follow the Dudley LSOA naming pattern.
Compare the LSOA codes with the cleaned economic activity dataset
to confirm that both datasets cover the same neighbourhoods.

In [12]:
# Developed with assistance from ChatGPT.

# Load the cleaned economic activity dataset for comparison.
economic_df = pd.read_csv(
    "cleaned_datasets/dudley_economic_activity_cleaned.csv"
)

# Check that all names follow the Dudley LSOA naming pattern.
dudley_names = disability_clean["LSOA_Name"].str.fullmatch(
    r"Dudley \d{3}[A-Z]",
    na=False
)

# Compare the sets of neighbourhood codes.
disability_codes = set(disability_clean["LSOA_Code"])
economic_codes = set(economic_df["LSOA_Code"])

print(f"Names matching Dudley LSOA pattern: {dudley_names.sum()}")
print(f"Disability LSOA codes: {len(disability_codes)}")
print(f"Economic activity LSOA codes: {len(economic_codes)}")
print(
    "Only in disability:",
    sorted(disability_codes - economic_codes)
)
print(
    "Only in economic activity:",
    sorted(economic_codes - disability_codes)
)

Names matching Dudley LSOA pattern: 203
Disability LSOA codes: 203
Economic activity LSOA codes: 203
Only in disability: []
Only in economic activity: []


### Findings

- All 203 area names match the Dudley LSOA naming pattern.
- Both datasets contain 203 unique LSOA codes.
- No codes appear exclusively in either dataset.
- The disability dataset covers exactly the same neighbourhoods
  as the cleaned economic activity dataset.
- LSOA_Code can therefore be used for matching these datasets
  during the later merging stage.

---

## 9. Validate population counts

Check the five count columns for non-numeric values, missing values,
negative numbers and fractional values before converting them to integers.

Population counts should be non-negative whole numbers.

## Ethical Considerations When Validating Disability Population Counts

Validating the five disability‑related population count columns is an essential ethical step in preparing this dataset.  
Because these values describe real residents aged 16 years and over — including those with activity‑limiting disabilities — ensuring numeric integrity is critical for fair, accurate, and responsible analysis.

### Why Ethical Validation Matters

Population counts in this dataset represent:
- Residents with disabilities  
- Residents limited “a lot”  
- Residents limited “a little”  
- Residents not disabled  
- The total population aged 16+  

These categories are used to understand:
- Health limitations  
- Barriers to employment  
- Social inclusion  
- Local support needs  
- Neighbourhood‑level inequalities  

Any errors in these counts can lead to harmful misinterpretations.

### Ethical Risks if Numeric Validation Is Ignored

If numeric validation is skipped or performed incorrectly, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  Non‑numeric or missing values may make an area appear more or less affected by disability than it truly is.

- **Incorrect totals**  
  Negative or fractional values can distort disability prevalence and mislead dashboards or reports.

- **Unfair comparisons**  
  Neighbourhoods may appear disadvantaged or advantaged due to data inconsistencies rather than real conditions.

- **Broken dataset joins**  
  Invalid numeric values can cause errors when merging with qualifications or economic activity datasets.

- **Loss of transparency**  
  Users may not understand why certain values appear inconsistent or contradictory.

### Ethical Safeguards Used in This Step

This validation step includes several responsible practices:

- **Selecting only the five population count columns**  
  Ensures that checks focus on the values most likely to affect analysis.

- **Using `pd.to_numeric` with coercion**  
  Safely identifies non‑numeric values without altering the dataset.

- **Checking for missing or non‑numeric values**  
  Confirms that all counts are present and valid.

- **Checking for negative values**  
  Ensures that disability counts do not fall below zero, which would be impossible and misleading.

- **Checking for fractional values**  
  Confirms that all counts represent whole people, not partial or rounded values.

- **Summarising issues clearly**  
  Supports transparency and allows analysts to verify data integrity before conversion.

### Why This Matters for Responsible Interpretation

Disability data is used to:
- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Incorrect handling of population counts can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By validating population counts carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [13]:
# Developed with assistance from ChatGPT.

# Select the five population count columns.
count_columns = [
    "Total_16_Plus",
    "Disabled_16_Plus",
    "Limited_A_Lot_16_Plus",
    "Limited_A_Little_16_Plus",
    "Not_Disabled_16_Plus"
]

# Check numeric conversion without changing the cleaning dataframe.
numeric_counts = disability_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Summarise issues separately for each column.
count_checks = pd.DataFrame({
    "Missing_or_non_numeric": numeric_counts.isna().sum(),
    "Negative_values": numeric_counts.lt(0).sum(),
    "Fractional_values": (
        numeric_counts.notna() & numeric_counts.mod(1).ne(0)
    ).sum()
})

count_checks

,Missing_or_non_numeric,Negative_values,Fractional_values
Total_16_Plus,0,0,0
Disabled_16_Plus,0,0,0
Limited_A_Lot_16_Plus,0,0,0
Limited_A_Little_16_Plus,0,0,0
Not_Disabled_16_Plus,0,0,0


### Findings

- All five count columns contain valid numeric values.
- No missing, non-numeric, negative or fractional values were found.
- The counts can be converted to integers without rounding or
  changing their values.
- No count corrections are needed based on these checks.

---

## 10. Convert population counts to integers

Convert the five validated count columns to integer data types.
This removes unnecessary decimal places without changing the counts.

In [14]:
# Developed with assistance from ChatGPT.

# Convert the validated whole-number counts to integers.
disability_clean[count_columns] = (
    disability_clean[count_columns].astype("int64")
)

# Confirm the resulting data types and preview five rows.
print(disability_clean[count_columns].dtypes)
disability_clean.head()

Total_16_Plus               int64
Disabled_16_Plus            int64
Limited_A_Lot_16_Plus       int64
Limited_A_Little_16_Plus    int64
Not_Disabled_16_Plus        int64
dtype: object


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus
0,Dudley 001A,E01009763,1640,350,154,196,1290
1,Dudley 001B,E01009764,1264,261,121,140,1003
2,Dudley 001C,E01009767,1268,296,145,151,972
3,Dudley 001D,E01009769,1428,376,208,168,1052
4,Dudley 001E,E01009770,1213,278,136,142,935


### Findings

- All five population count columns now use the int64 data type.
- The preview displays whole numbers without decimal places.
- Conversion preserved the validated counts; no rounding was needed.
- LSOA names and codes were unchanged.

---

## 11. Check consistency between population counts

Check whether the two activity-limitation subcategories sum to the
disabled total, and whether disabled plus not disabled equals the
total population aged 16 and over.

Report any differences for inspection without changing source counts.

## Ethical Considerations When Checking Consistency Between Disability Population Counts

Checking the internal consistency of disability population counts is a crucial ethical step.  
Because these values describe real residents aged 16 years and over — including those with activity‑limiting disabilities — ensuring that the subcategory totals align with the overall disability and population counts is essential for fair, accurate, and responsible analysis.

### Why Ethical Consistency Checks Matter

The disability dataset includes three related groups:

- Residents limited **a lot**  
- Residents limited **a little**  
- Total residents **with disabilities**  
- Residents **not disabled**  
- Total population aged 16+  

These categories must align logically.  
If they do not, it may indicate structural issues such as:

- Misclassification  
- Export errors  
- Disclosure‑control adjustments  
- Missing or corrupted values  

Ensuring consistency protects against misinterpretation and supports ethical use of sensitive health‑related data.

### Ethical Risks if Consistency Checks Are Ignored

If these relationships are not validated, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  Incorrect totals may exaggerate or understate disability prevalence.

- **Distorted comparisons**  
  Dashboards or reports may show misleading patterns if subcategories do not sum correctly.

- **Incorrect policy insights**  
  Resource allocation or support planning may be based on inaccurate disability counts.

- **Loss of transparency**  
  Users may not understand why totals appear contradictory or inconsistent.

- **Broken dataset joins**  
  Inconsistent totals can cause confusion when merging with qualifications or economic activity datasets.

### Ethical Safeguards Used in This Step

This consistency check includes several responsible practices:

- **Comparing activity‑limitation subcategories with the disabled total**  
  Ensures that “limited a lot” + “limited a little” equals the total number of disabled residents.

- **Comparing disabled and not disabled totals with the population total**  
  Confirms that the dataset’s structure reflects real population logic.

- **Reporting differences without altering values**  
  Preserves data integrity and supports transparent investigation.

- **Previewing affected rows**  
  Allows analysts to inspect any inconsistencies before deciding whether further action is needed.

- **Maintaining the cleaned dataset separately**  
  Ensures that raw data remains unchanged and auditable.

### Why This Matters for Responsible Interpretation

Disability data is used to:

- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Incorrect handling of internal totals can lead to:

- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By checking population‑count consistency carefully and transparently, this project ensures that Dudley’s disability data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [15]:
# Developed with assistance from ChatGPT.

# Compare disability subcategories with the disabled total.
disabled_difference = (
    disability_clean["Limited_A_Lot_16_Plus"]
    + disability_clean["Limited_A_Little_16_Plus"]
    - disability_clean["Disabled_16_Plus"]
)

# Compare the two disability groups with the population total.
total_difference = (
    disability_clean["Disabled_16_Plus"]
    + disability_clean["Not_Disabled_16_Plus"]
    - disability_clean["Total_16_Plus"]
)

print(
    "Rows where disability subcategories do not match:",
    disabled_difference.ne(0).sum()
)
print(
    "Rows where disability groups do not match the total:",
    total_difference.ne(0).sum()
)

# Preview any records with differences.
difference_rows = disability_clean.loc[
    disabled_difference.ne(0) | total_difference.ne(0)
]
difference_rows.head()

Rows where disability subcategories do not match: 0
Rows where disability groups do not match the total: 0


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus


### Findings

- For all 203 neighbourhoods, the two activity-limitation
  subcategories sum exactly to the disabled total.
- Disabled plus not disabled equals the total population aged
  16 and over in every row.
- No inconsistencies were found, so the differences table is empty.
- No adjustments to the source counts are needed.

---

## 12. Save the cleaned dataset

Save the validated dataset as a CSV without the dataframe index.
The raw source file remains unchanged.

## Ethical Considerations When Saving the Cleaned Disability Dataset

Saving the cleaned disability dataset is a crucial ethical step in the workflow.  
Because this dataset contains sensitive information about residents aged 16 years and over — including activity‑limiting disability categories — it is essential to preserve accuracy, protect data integrity, and maintain transparency when exporting the final validated file.

### Why Ethical Care Is Needed Here

The cleaned dataset now contains:
- 203 verified neighbourhood (LSOA) records  
- No missing values  
- No duplicate rows  
- No repeated LSOA codes  
- Fully validated disability counts  
- Consistent internal totals across all categories  

This dataset will be used for:
- Cross‑dataset merging  
- Visualisations  
- Statistical summaries  
- Public‑facing dashboards  
- Policy‑relevant insights  

If the saved file contains errors, missing values, or unintended changes, the consequences can cascade through the entire project and lead to **misleading or unfair conclusions** about neighbourhoods.

### Ethical Risks if Saving Is Done Incorrectly

If the dataset is saved or reloaded incorrectly, several ethical risks arise:

- **Loss of data integrity**  
  Exporting may accidentally drop rows, alter values, or change column types.

- **Silent introduction of missing values**  
  Encoding or formatting issues may convert valid values to blanks or `NaN`.

- **Corrupted or duplicated records**  
  Incorrect saving may introduce repeated rows or misaligned identifiers.

- **Misalignment of LSOA codes**  
  If identifiers change during export, dataset joins may fail or misrepresent communities.

- **Reduced transparency**  
  Without verification, users cannot be confident that the saved dataset matches the cleaned version.

### Ethical Safeguards Used in This Step

This saving process includes several responsible practices:

- **Saving without the DataFrame index**  
  Prevents accidental introduction of non‑data columns that could confuse downstream users.

- **Using a dedicated `cleaned_datasets` folder**  
  Keeps cleaned data separate from raw data, supporting transparency and auditability.

- **Preserving the raw dataset unchanged**  
  Maintains a clear audit trail and supports ethical data governance.

- **Confirming the save location and dataset dimensions**  
  Provides immediate feedback that the export was successful and complete.

- **Retaining validated counts exactly as they are**  
  No adjustments or rounding were applied, ensuring the dataset remains faithful to the source.

### Why This Matters for Responsible Interpretation

Disability data is used to:
- Understand health limitations  
- Identify support needs  
- Inform local policy decisions  
- Allocate resources fairly  
- Analyse employment barriers  

Errors introduced during saving can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By saving and verifying the cleaned dataset carefully, this project ensures that Dudley’s disability data is preserved **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [16]:
# Developed with assistance from ChatGPT.

# Save the cleaned dataset without an extra index column.
output_path = "cleaned_datasets/dudley_disability_16_cleaned.csv"
disability_clean.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(disability_clean)}")
print(f"Columns: {disability_clean.shape[1]}")

Saved: cleaned_datasets/dudley_disability_16_cleaned.csv
Rows: 203
Columns: 7


---

## Conclusion

The disability dataset has been cleaned and validated for 203 Dudley
LSOAs, covering residents aged 16 years and over.

Eight non-data rows were removed: four blank records and four source-note
rows. All seven columns were renamed, and the five population count
columns were converted to integers without changing their values.

Checks confirmed:
- No missing values, duplicate rows or repeated LSOA codes.
- Matching LSOA coverage with the cleaned economic activity dataset.
- Non-negative whole-number counts.
- Consistent disability subcategory and population totals.

The cleaned output is:
`cleaned_datasets/dudley_disability_16_cleaned.csv`.

The raw source file remains unchanged. The cleaned data is ready for
later merging and analysis. Neighbourhood disability counts must not
be interpreted as evidence of an individual's ability or willingness
to work.

## Ethical Considerations in the Final Conclusions

The cleaning and validation of the disability dataset is now complete, and all checks confirm that the data is reliable, internally consistent, and ethically suitable for further analysis.  
Because this dataset represents real residents aged 16 years and over — including those with activity‑limiting disabilities — the conclusions must reflect not only technical correctness but also the ethical responsibilities upheld throughout the process.

### Ethical Significance of the Cleaning Outcome

The final cleaned dataset contains:
- 203 verified neighbourhood (LSOA) records  
- No missing values  
- No duplicate rows  
- No repeated LSOA codes  
- Fully validated disability counts  
- Correct internal totals across all categories  
- All population counts stored as whole numbers (`int64`)  
- Preserved and clearly labelled geographic identifiers (`LSOA_Name`, `LSOA_Code`)  

These results demonstrate that the dataset is structurally sound, internally coherent, and ethically robust.

### Ethical Risks Avoided Through Careful Validation

The workflow avoided several potential harms:

- **Misrepresenting neighbourhoods** through missing or inconsistent disability counts  
- **Inflating or deflating disability prevalence** due to duplicated or corrupted records  
- **Misaligned geographic identifiers** that could distort cross‑dataset joins  
- **Silent errors during export** that could mislead dashboards or reports  
- **Unfair comparisons** caused by inconsistent or contradictory values  

Avoiding these risks is essential when analysing health limitations, support needs, and employment barriers.

### Ethical Strengths of the Completed Workflow

The cleaning process demonstrates:

- **Transparency** — every transformation was documented and justified  
- **Accuracy** — all values were validated before conversion  
- **Fairness** — no LSOA was removed, duplicated, or misrepresented  
- **Accountability** — the saved dataset was verified against the cleaned DataFrame  
- **Respect for communities** — the raw dataset remains unchanged and auditable  
- **Consistency** — column names are clear, meaningful, and aligned with other datasets  

These practices ensure that the dataset can be used responsibly in later notebooks, dashboards, and reports.

### Ethical Conclusion

The cleaned disability dataset is now ethically robust and technically reliable.  
It preserves the integrity of Dudley’s LSOA‑level information, supports fair and accurate analysis, and provides a trustworthy foundation for exploring disability prevalence, activity‑limitation patterns, and employment barriers without misrepresenting or harming the communities described.



---